# 2부 자습 노트북 — 결정 트리 심화

본 노트북은 *직접 실행하며* 학습하는 자료이다. 셀을 위에서 아래로 *차례로 실행*하면 2부 이론 교재(`part2_tree_advanced_HARD_이론.md`)의 핵심 코드를 모두 돌려볼 수 있다.

**권장 선행 학습**: 3부 자습 노트북(`part3_bagging_RF_HARD_자습노트북.ipynb`)을 먼저 마친 후 본 노트북을 진행한다. 3부 0장이 트리의 *기초*를 다루고, 본 2부가 *심화 주제*를 다룬다.

**데이터셋**:
- **Ames** (2,930 × 82) — 강사 시범 (Ping)
- **Cereals** (77 × 16) — 학생 실습 (Pong)

## 환경 준비와 데이터 로딩

In [ ]:
# Colab 등에서 처음 한 번만 실행
# !pip install koreanize-matplotlib --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["axes.unicode_minus"] = False

URL_AMES   = "https://raw.githubusercontent.com/leina99-lab/classes/main/AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/data/AmesHousing.csv"
URL_CEREAL = "https://raw.githubusercontent.com/leina99-lab/classes/main/AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/data/Cereals.csv"

try:
    ames_raw   = pd.read_csv(URL_AMES)
    cereal_raw = pd.read_csv(URL_CEREAL)
except Exception:
    rng = np.random.default_rng(42)
    n = 2900
    ames_raw = pd.DataFrame({
        "Overall Qual": rng.integers(1, 11, n),
        "Gr Liv Area":  rng.integers(500, 4500, n),
        "Year Built":   rng.integers(1900, 2010, n),
        "1st Flr SF":   rng.integers(400, 2000, n),
        "2nd Flr SF":   rng.integers(0, 1500, n),
        "Total Bsmt SF": rng.integers(0, 1500, n),
        "Garage Cars":  rng.integers(0, 4, n),
    })
    ames_raw["SalePrice"] = (
        50000 + ames_raw["Overall Qual"] * 25000
        + ames_raw["Gr Liv Area"] * 60 + rng.normal(0, 20000, n)
    ).astype(int)
    cereal_raw = pd.DataFrame({
        "name":     [f"Cereal_{i}" for i in range(77)],
        "mfr":      rng.choice(list("KGNQACR"), 77),
        "type":     rng.choice(["C", "H"], 77),
        "calories": rng.integers(50, 160, 77),
        "rating":   rng.uniform(18, 95, 77).round(2),
    })

print(f"Ames:    {ames_raw.shape}")
print(f"Cereals: {cereal_raw.shape}")

1부의 표준 전처리 함수를 가져온다.

In [ ]:
def prepare_ames(df_in):
    df = df_in.copy()
    df = df.drop(columns=[c for c in ["Order", "PID"] if c in df.columns])
    for c in ["Pool QC", "Misc Feature", "Alley", "Fence", "Fireplace Qu",
              "Garage Qual", "Garage Cond", "Garage Finish", "Garage Type",
              "Bsmt Qual", "Bsmt Cond", "Bsmt Exposure",
              "BsmtFin Type 1", "BsmtFin Type 2", "Mas Vnr Type"]:
        if c in df.columns:
            df[c] = df[c].fillna("None")
    num = df.select_dtypes("number").columns
    df[num] = df[num].fillna(df[num].median())
    df = df[df["Gr Liv Area"] < 4000].copy()
    if all(c in df.columns for c in ["1st Flr SF", "2nd Flr SF", "Total Bsmt SF"]):
        df["Total SF"] = df["1st Flr SF"] + df["2nd Flr SF"] + df["Total Bsmt SF"]
    return df

ames = prepare_ames(ames_raw)
y_ames = np.log1p(ames["SalePrice"])
X_ames = ames.select_dtypes("number").drop(columns=["SalePrice"])

print(f"X_ames shape: {X_ames.shape}")
print(f"y_ames shape: {y_ames.shape}")

---
## 0장 왜 트리 심화가 필요한가

본 시리즈는 *부 번호 순*이 학습 순서와 일치하지 않는다. 트리 *기초*가 3부 0장에 포함되고, 본 2부는 *심화 주제 8개*를 다룬다.

본 부의 핵심 메시지 두 가지:
- 단일 트리도 잘 다듬으면 *R² 0.83*까지 도달 (3부 기본 0.77에서 끌어올림)
- 본 부의 매개변수는 *RF, GBM, XGBoost*에도 그대로 적용된다

---
## 1장 분할 알고리즘의 복잡도

sklearn의 트리 학습은 $O(d \cdot n \log n)$이다. 직접 측정해 본다.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.datasets import make_regression
import time

print(f"{'n':>8s}  {'학습 시간 (ms)':>15s}")
print("-" * 26)
times = []
sizes = [100, 500, 1000, 5000, 10000]
for n in sizes:
    X, y = make_regression(n_samples=n, n_features=10, random_state=42)
    t0 = time.time()
    tree = DecisionTreeRegressor(random_state=42)
    tree.fit(X, y)
    dt = (time.time() - t0) * 1000
    times.append(dt)
    print(f"{n:>8d}  {dt:>15.2f}")

print(f"\nn 100→10000 (100배): 시간 {times[-1]/times[0]:.1f}배 증가")
print(f"이론적 O(n log n) 예측: 약 200배 (n²이라면 10,000배)")

### 시각화 — O(n log n) 직접 확인

In [ ]:
n_arr = np.array(sizes)
theory = n_arr * np.log(n_arr)
theory = theory * (times[0] / theory[0])  # 첫 점에 맞춤

quad = n_arr ** 2
quad = quad * (times[0] / quad[0])

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sizes, times, "o-", color="#1F3A5F", linewidth=2.5, markersize=10, label="실제 측정")
ax.plot(sizes, theory, "--", color="#C0392B", linewidth=2, label="이론적 O(n log n)")
ax.plot(sizes, quad, ":", color="#7B8794", linewidth=2, alpha=0.5, label="비교: O(n^2)")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("샘플 수 n")
ax.set_ylabel("학습 시간 (ms, 로그)")
ax.legend()
ax.set_title("결정 트리 학습 시간 — 실제로 O(n log n) 비례")
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

---
## 2장 비용 복잡도 가지치기 — ccp_alpha

학습 후 *불필요한 가지를 잘라내는* 사후 가지치기. alpha가 클수록 트리가 작아진다.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X_tr, X_te, y_tr, y_te = train_test_split(X_ames, y_ames, test_size=0.3, random_state=42)

print(f"{'alpha':>10s}  {'leaves':>8s}  {'depth':>6s}  {'test R²':>10s}")
print("-" * 40)
for a in [0.0, 0.00005, 0.0001, 0.0005, 0.001, 0.005]:
    t = DecisionTreeRegressor(ccp_alpha=a, random_state=42)
    t.fit(X_tr, y_tr)
    r2 = r2_score(y_te, t.predict(X_te))
    print(f"{a:>10.5f}  {t.get_n_leaves():>8d}  {t.get_depth():>6d}  {r2:>10.4f}")

print("\n무가지치기(α=0): leaves 1988, R² 0.78 (과적합)")
print("최적 (α≈0.0001):  leaves 89,   R² 0.83 (튜닝 효과!)")

### alpha 경로 — 트리가 어떻게 축소되는지 추적

In [ ]:
tree_full = DecisionTreeRegressor(random_state=42)
tree_full.fit(X_tr, y_tr)
path = tree_full.cost_complexity_pruning_path(X_tr, y_tr)

print(f"alpha 경로 길이: {len(path.ccp_alphas)}개")
print(f"alpha 범위: {path.ccp_alphas[0]:.2e} ~ {path.ccp_alphas[-2]:.2e}")

### 시각화 — alpha 따라 R²와 잎 수

In [ ]:
alphas = np.geomspace(1e-5, 0.05, 20)
train_r2, test_r2, n_leaves = [], [], []
for a in alphas:
    t = DecisionTreeRegressor(ccp_alpha=a, random_state=42)
    t.fit(X_tr, y_tr)
    train_r2.append(r2_score(y_tr, t.predict(X_tr)))
    test_r2.append(r2_score(y_te, t.predict(X_te)))
    n_leaves.append(t.get_n_leaves())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(alphas, train_r2, "o-", color="#1F3A5F", label="학습 R²")
axes[0].plot(alphas, test_r2,  "s-", color="#C0392B", label="검증 R²")
axes[0].set_xscale("log")
axes[0].set_xlabel("ccp_alpha")
axes[0].set_ylabel("R²")
axes[0].legend()
axes[0].set_title("ccp_alpha에 따른 R² 변화")
axes[0].grid(alpha=0.3)

axes[1].plot(alphas, n_leaves, "o-", color="#1F3A5F")
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("ccp_alpha")
axes[1].set_ylabel("잎 수 (로그)")
axes[1].set_title("ccp_alpha에 따른 트리 크기")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3장 매개변수 미세 조정

사전 가지치기 매개변수 4종의 효과를 직접 측정한다.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score

# (1) max_depth
print("=== max_depth ===")
for d in [3, 5, 7, 10, None]:
    t = DecisionTreeRegressor(max_depth=d, random_state=42)
    r2 = cross_val_score(t, X_ames, y_ames, cv=5, scoring="r2", n_jobs=-1).mean()
    print(f"  max_depth={str(d):>6s}: R² = {r2:.4f}")

In [ ]:
# (2) min_samples_leaf
print("=== min_samples_leaf ===")
for n in [1, 5, 10, 20, 50]:
    t = DecisionTreeRegressor(min_samples_leaf=n, random_state=42)
    r2 = cross_val_score(t, X_ames, y_ames, cv=5, scoring="r2", n_jobs=-1).mean()
    print(f"  leaf={n:>3d}: R² = {r2:.4f}")

print()
print("→ min_samples_leaf=10이 단일 매개변수로 R² 0.82 도달!")

### GridSearchCV — 4종 매개변수 조합 탐색

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "max_depth": [5, 7, 10],
    "min_samples_leaf": [5, 10, 20],
    "ccp_alpha": [0, 1e-5, 1e-4]
}

grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid, cv=5, scoring="r2", n_jobs=-1
)
grid.fit(X_ames, y_ames)
print(f"최적 매개변수: {grid.best_params_}")
print(f"최적 CV R²:    {grid.best_score_:.4f}")

---
## 4장 단조성 제약 — 도메인 지식을 트리에 부과

`Overall Qual → SalePrice`는 *명백한 단조 증가*다. sklearn 1.4+ 에서 `monotonic_cst` 매개변수로 이를 강제할 수 있다.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score

# Overall Qual의 컬럼 인덱스
cols = X_ames.columns.tolist()
qual_idx = cols.index("Overall Qual") if "Overall Qual" in cols else 0

# 제약 벡터 — 모두 0 (제약 없음)으로 초기화 후 Qual만 1로
constraints = [0] * len(cols)
constraints[qual_idx] = 1   # 단조 증가

tree_free = DecisionTreeRegressor(max_depth=8, random_state=42)
tree_mono = DecisionTreeRegressor(max_depth=8, monotonic_cst=constraints, random_state=42)

r2_free = cross_val_score(tree_free, X_ames, y_ames, cv=5, scoring="r2", n_jobs=-1).mean()
r2_mono = cross_val_score(tree_mono, X_ames, y_ames, cv=5, scoring="r2", n_jobs=-1).mean()

print(f"제약 없음:        R² = {r2_free:.4f}")
print(f"Qual 단조 증가:   R² = {r2_mono:.4f}")
print(f"\n→ 제약이 R²를 약간 올렸다 (잡음 따라가는 과적합 방어 효과)")

### 단조성 제약의 시각적 검증 — 1차원 예시

In [ ]:
rng = np.random.default_rng(42)
n_pts = 200
x = np.linspace(0, 10, n_pts)
y_true = np.where(x < 5, x*1.5, 7.5 + (x-5)*0.5)
y = y_true + rng.normal(0, 1.5, n_pts)
X = x.reshape(-1, 1)

tree_free = DecisionTreeRegressor(max_depth=4, random_state=42)
tree_free.fit(X, y)
tree_mono = DecisionTreeRegressor(max_depth=4, monotonic_cst=[1], random_state=42)
tree_mono.fit(X, y)

x_grid = np.linspace(0, 10, 200).reshape(-1, 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(x, y, c="#7B8794", s=20, alpha=0.5)
axes[0].plot(x_grid, tree_free.predict(x_grid), color="#C0392B", linewidth=2.5, label="제약 없는 트리")
axes[0].set_title("제약 없음 — 잡음 따라가다 비단조 구간 생김")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].scatter(x, y, c="#7B8794", s=20, alpha=0.5)
axes[1].plot(x_grid, tree_mono.predict(x_grid), color="#1F3A5F", linewidth=2.5, label="단조 증가 제약")
axes[1].set_title("monotonic_cst=[1] — 단조 증가만 허용")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5장 다중 출력 트리 — 한 트리로 여러 타깃 동시 예측

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# 가상 데이터: 입력 10차원, 타깃 2개 (공통 구조 공유)
X, y_single = make_regression(n_samples=500, n_features=10, random_state=42)
rng = np.random.default_rng(0)
y_multi = np.column_stack([y_single, y_single * 2 + rng.normal(0, 5, 500)])

print(f"X shape:       {X.shape}")
print(f"y_multi shape: {y_multi.shape}")

X_tr, X_te, y_tr, y_te = train_test_split(X, y_multi, test_size=0.3, random_state=42)

# 다중 출력 — 별다른 설정 없이
tree = DecisionTreeRegressor(max_depth=5, random_state=42)
tree.fit(X_tr, y_tr)
pred = tree.predict(X_te)

print(f"\n예측 shape: {pred.shape}")
print(f"y1 R²: {r2_score(y_te[:,0], pred[:,0]):.4f}")
print(f"y2 R²: {r2_score(y_te[:,1], pred[:,1]):.4f}")

---
## 6장 범주형 변수의 트리 분할

라벨 인코딩 vs 원-핫 vs 네이티브 처리 — 세 방법을 Cereals에서 비교한다.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score

# 데이터 준비
df = cereal_raw.copy()
num = df.select_dtypes("number").columns
df[num] = df[num].fillna(df[num].median())

# (1) 라벨 인코딩
df_le = df.copy()
for c in ["mfr", "type"]:
    df_le[c] = LabelEncoder().fit_transform(df_le[c])
X_le = df_le.drop(columns=["name", "rating"])

# (2) 원-핫 인코딩
X_oh = pd.get_dummies(df.drop(columns=["name", "rating"]),
                       columns=["mfr", "type"], drop_first=True)

# (3) 네이티브 (category 타입)
df_cat = df.drop(columns=["name"]).copy()
df_cat["mfr"] = df_cat["mfr"].astype("category")
df_cat["type"] = df_cat["type"].astype("category")
X_cat = df_cat.drop(columns=["rating"])

y = df["rating"]

print(f"{'방법':<25s}  {'R²':>10s}")
print("-" * 38)

r2_le = cross_val_score(GradientBoostingRegressor(random_state=42, n_estimators=100),
                         X_le, y, cv=5, scoring="r2").mean()
print(f"{'라벨 + GBM':<25s}  {r2_le:>10.4f}")

r2_oh = cross_val_score(GradientBoostingRegressor(random_state=42, n_estimators=100),
                         X_oh, y, cv=5, scoring="r2").mean()
print(f"{'원-핫 + GBM':<25s}  {r2_oh:>10.4f}")

r2_cat = cross_val_score(HistGradientBoostingRegressor(
                            categorical_features="from_dtype",
                            random_state=42, max_iter=100),
                          X_cat, y, cv=5, scoring="r2").mean()
print(f"{'네이티브 HistGB':<25s}  {r2_cat:>10.4f}")

print("\n→ Cereals 같은 작은 데이터에서는 원-핫이 가장 좋다.")

---
## 7장 트리 시각화 도구

`plot_tree`, `export_text`로 트리의 구조를 직접 본다.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree

# 보기 좋게 작은 트리
tree_small = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_small.fit(X_ames, y_ames)

fig, ax = plt.subplots(figsize=(20, 10), dpi=100)
plot_tree(tree_small,
          feature_names=X_ames.columns,
          filled=True, rounded=True,
          fontsize=9, ax=ax)
plt.tight_layout()
plt.show()

print(f"\n루트 분할 변수: {X_ames.columns[tree_small.tree_.feature[0]]}")
print(f"루트 임계값:    {tree_small.tree_.threshold[0]:.2f}")

### export_text — 텍스트로 출력

In [ ]:
from sklearn.tree import export_text

text_rules = export_text(tree_small,
                          feature_names=X_ames.columns.tolist(),
                          max_depth=3)
print(text_rules)

### 변수 중요도 — 큰 트리 요약

In [ ]:
tree_big = DecisionTreeRegressor(max_depth=10, random_state=42)
tree_big.fit(X_ames, y_ames)

imp = pd.Series(tree_big.feature_importances_, index=X_ames.columns)
top10 = imp.nlargest(10)
print("변수 중요도 상위 10개:")
print(top10.round(4))

---
## 8장 단일 트리의 한계 — 앙상블로의 다리

본 부에서 단일 트리를 *R² 0.83*까지 끌어올렸다. 그러나 *앙상블*에는 못 미친다.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models = {
    "단일 트리 (기본)":   DecisionTreeRegressor(random_state=42),
    "단일 트리 (튜닝)":   DecisionTreeRegressor(max_depth=7, min_samples_leaf=10,
                                                 ccp_alpha=1e-4, random_state=42),
    "랜덤 포레스트":      RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "GBM":                GradientBoostingRegressor(n_estimators=100, random_state=42),
}

print(f"{'모델':<25s}  {'CV R²':>10s}")
print("-" * 38)
for name, m in models.items():
    r2 = cross_val_score(m, X_ames, y_ames, cv=5, scoring="r2", n_jobs=-1).mean()
    print(f"{name:<25s}  {r2:>10.4f}")

print("\n→ 단일 트리의 한계는 약 0.83. 그 이상은 앙상블의 영역.")

---
## 마무리

본 노트북에서 *직접 실행*한 8가지 심화 도구를 한 표로 정리한다.

| 장 | 핵심 도구 |
|---|---|
| 1장 분할 복잡도 | `time.time` 측정, O(n log n) 검증 |
| 2장 가지치기 | `ccp_alpha`, `cost_complexity_pruning_path` |
| 3장 매개변수 | `max_depth`, `min_samples_leaf`, `GridSearchCV` |
| 4장 단조성 제약 | `monotonic_cst=[1, 0, ...]` |
| 5장 다중 출력 | `y_multi = np.column_stack([...])` |
| 6장 범주형 분할 | `pd.get_dummies` vs `categorical_features="from_dtype"` |
| 7장 시각화 | `plot_tree`, `export_text`, `feature_importances_` |
| 8장 한계 | 단일 트리 0.83 → 앙상블 0.88+ |

### 다음 단계

4부로 이동하여 **AdaBoost**를 자습한다. *지수 손실*에 갇힌 부스팅이 *왜 캘리포니아에서 무너지는지* 본격적으로 본다.